In [58]:
import geopandas as gpd
import zipfile
import os
from pathlib import Path

# Path to your KMZ file
kmz_path = r"./TN Forest Boundary.kmz"

# KMZ is a zipped KML file, so we need to extract it first
extract_dir = r"./TN_Forest_Extracted"
os.makedirs(extract_dir, exist_ok=True)

# Extract KMZ contents
with zipfile.ZipFile(kmz_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
    print("Extracted files:")
    for file in zip_ref.namelist():
        print(f"  - {file}")


Extracted files:
  - doc.kml


In [59]:
import geopandas as gpd
import fiona
from pathlib import Path

# Find the KML file (usually doc.kml)
kml_files = list(Path(extract_dir).glob("*.kml"))
print(f"\nFound KML files: {kml_files}")

# Read the KML file with geopandas
if kml_files:
    kml_path = str(kml_files[0])
    
    # Read all layers to see the structure
    layers = fiona.listlayers(kml_path)
    print(f"\nLayers in KML: {layers}")
    
    # Read each layer to understand structure
    for layer in layers:
        try:
            gdf = gpd.read_file(kml_path, driver='KML', layer=layer)
            print(f"\n--- Layer: {layer} ---")
            print(f"Number of features: {len(gdf)}")
            print(f"Columns: {gdf.columns.tolist()}")
            print(f"\nFirst few rows:")
            print(gdf.head())
            print(f"\nGeometry types: {gdf.geometry.type.unique()}")
        except Exception as e:
            print(f"Error reading layer {layer}: {e}")
else:
    print("No KML files found in extracted directory")


Found KML files: [WindowsPath('TN_Forest_Extracted/doc.kml')]

Layers in KML: ['Point Features', 'Area Features']

--- Layer: Point Features ---
Number of features: 34
Columns: ['Name', 'Description', 'geometry']

First few rows:
                             Name           Description  \
0   Vallanadu BlackBuck Sanctuary  BOUNDARY_FOREST_ZONE   
1                        Nadugani  BOUNDARY_FOREST_ZONE   
2                        O'valley  BOUNDARY_FOREST_ZONE   
3                 Sandynulla-ooty  BOUNDARY_FOREST_ZONE   
4  Nilagiri Eastern Part & Cooner  BOUNDARY_FOREST_ZONE   

                        geometry  
0   POINT Z (77.87522 8.69827 0)  
1  POINT Z (76.38635 11.43921 0)  
2   POINT Z (76.5567 11.42646 0)  
3  POINT Z (76.65855 11.44113 0)  
4    POINT Z (76.871 11.52646 0)  

Geometry types: ['Point']

--- Layer: Area Features ---
Number of features: 99
Columns: ['Name', 'Description', 'geometry']

First few rows:
  Name           Description  \
0       BOUNDARY_FOREST_ZONE  

In [51]:
import pandas as pd

# Read all KML layers and combine into one GeoDataFrame
gdf_layers = []

for lyr in layers:
    layer_gdf = gpd.read_file(kml_path, driver="KML", layer=lyr)
    layer_gdf["layer"] = lyr
    gdf_layers.append(layer_gdf)

gdf_all = gpd.GeoDataFrame(
    pd.concat(gdf_layers, ignore_index=True),
    crs=gdf_layers[0].crs if gdf_layers else None
)

print(gdf_all[["layer", "Name", "geometry"]].head())
print(f"Total features: {len(gdf_all)}")

            layer                            Name  \
0  Point Features   Vallanadu BlackBuck Sanctuary   
1  Point Features                        Nadugani   
2  Point Features                        O'valley   
3  Point Features                 Sandynulla-ooty   
4  Point Features  Nilagiri Eastern Part & Cooner   

                        geometry  
0   POINT Z (77.87522 8.69827 0)  
1  POINT Z (76.38635 11.43921 0)  
2   POINT Z (76.5567 11.42646 0)  
3  POINT Z (76.65855 11.44113 0)  
4    POINT Z (76.871 11.52646 0)  
Total features: 133


In [52]:
gdf_all.head()

,Name,Description,geometry,layer
0,Vallanadu BlackBuck Sanctuary,BOUNDARY_FOREST_ZONE,POINT Z (77.87522 8.69827 0),Point Features
1,Nadugani,BOUNDARY_FOREST_ZONE,POINT Z (76.38635 11.43921 0),Point Features
2,O'valley,BOUNDARY_FOREST_ZONE,POINT Z (76.5567 11.42646 0),Point Features
3,Sandynulla-ooty,BOUNDARY_FOREST_ZONE,POINT Z (76.65855 11.44113 0),Point Features
4,Nilagiri Eastern Part & Cooner,BOUNDARY_FOREST_ZONE,POINT Z (76.871 11.52646 0),Point Features


In [53]:
gdf = gdf_all.copy()

In [46]:
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Polygon
from shapely.ops import polygonize, unary_union

def line_to_polygon_if_closed(geom):
    if geom is None or geom.is_empty:
        return geom

    # Case 1: single LineString that is a closed ring
    if isinstance(geom, LineString):
        if geom.is_ring and len(geom.coords) >= 4:
            poly = Polygon(geom)
            if poly.is_valid and not poly.is_empty and poly.area > 0:
                return poly
        return geom

    # Case 2: MultiLineString that may form one or more closed polygons
    if isinstance(geom, MultiLineString):
        polys = list(polygonize(unary_union(geom)))
        if len(polys) == 1:
            return polys[0]          # one polygon formed
        elif len(polys) > 1:
            return unary_union(polys) # multipolygon/merged geometry
        return geom

    return geom

# Apply to your GeoDataFrame
gdf["geometry"] = gdf["geometry"].apply(line_to_polygon_if_closed)

In [54]:
print(gdf.geom_type.value_counts())
print("Invalid geometries:", (~gdf.is_valid).sum())

LineString    61
Polygon       38
Point         34
Name: count, dtype: int64
Invalid geometries: 2


In [44]:
gdf.head()

,Name,Description,geometry
0,,BOUNDARY_FOREST_ZONE,"LINESTRING Z (76.51568 11.26905 0, 76.52606 11..."
1,,BOUNDARY_FOREST_ZONE,"POLYGON Z ((80.10798 12.89367 0, 80.10887 12.8..."
2,,BOUNDARY_FOREST_ZONE,"LINESTRING Z (76.68758 11.01338 0, 76.71188 11..."
3,,BOUNDARY_FOREST_ZONE,"LINESTRING Z (77.43341 11.55065 0, 77.46276 11..."
4,,BOUNDARY_FOREST_ZONE,"POLYGON Z ((77.87037 8.69231 0, 77.87044 8.692..."


In [61]:
from shapely.geometry import LineString, MultiLineString, Polygon, MultiPolygon, GeometryCollection
from shapely.ops import unary_union, polygonize_full, snap
from shapely import make_valid
from shapely.errors import GEOSException
import pandas as pd

def _flatten_lines(geom):
    """Return individual LineStrings from any line-like geometry."""
    if geom is None or geom.is_empty:
        return []

    if isinstance(geom, LineString):
        return [geom]
    if isinstance(geom, MultiLineString):
        return [g for g in geom.geoms if not g.is_empty]
    if isinstance(geom, GeometryCollection):
        out = []
        for g in geom.geoms:
            out.extend(_flatten_lines(g))
        return out
    return []

def _flatten_polygons(geom):
    """Return individual Polygons from polygon-like geometry."""
    if geom is None or geom.is_empty:
        return []

    if isinstance(geom, Polygon):
        return [geom]
    if isinstance(geom, MultiPolygon):
        return [g for g in geom.geoms if not g.is_empty]
    if isinstance(geom, GeometryCollection):
        out = []
        for g in geom.geoms:
            out.extend(_flatten_polygons(g))
        return out
    return []

def _safe_make_valid(geom):
    if geom is None or geom.is_empty:
        return geom
    try:
        return make_valid(geom)
    except Exception:
        try:
            return geom.buffer(0)
        except Exception:
            return geom

def _safe_intersection_area(a, b):
    try:
        return a.intersection(b).area
    except GEOSException:
        try:
            return _safe_make_valid(a).intersection(_safe_make_valid(b)).area
        except Exception:
            try:
                return a.buffer(0).intersection(b.buffer(0)).area
            except Exception:
                return 0.0

def _safe_intersects(a, b):
    try:
        return a.intersects(b)
    except GEOSException:
        return _safe_intersection_area(a, b) > 0

def _build_full_polygons(lines, snap_tolerance_m, min_area_m2):
    """Polygonize from line network, returning polygons and leftover linework."""
    if not lines:
        return [], [], {"network_lines": 0, "polygonized": 0, "leftover": 0}

    merged = unary_union(lines)
    noded = snap(merged, merged, snap_tolerance_m) if snap_tolerance_m > 0 else merged

    polys_gc, dangles_gc, cuts_gc, invalid_gc = polygonize_full(noded)

    poly_geoms = [
        p for p in getattr(polys_gc, "geoms", [])
        if (not p.is_empty) and p.is_valid and p.area >= min_area_m2
    ]

    leftover_geoms = []
    for gc in (dangles_gc, cuts_gc, invalid_gc):
        for g in getattr(gc, "geoms", []):
            leftover_geoms.extend(_flatten_lines(g))

    stats = {
        "network_lines": len(lines),
        "polygonized": len(poly_geoms),
        "leftover": len(leftover_geoms),
    }
    return poly_geoms, leftover_geoms, stats

def convert_lines_and_fuse_with_polygons(
    gdf_input,
    snap_tolerance_m=8.0,
    min_area_m2=25.0,
    dedupe_overlap_ratio=0.98,
    area_growth_ratio=1.05,
):
    """
    Stage 1: Build polygons from line-only network and keep unused tails.
    Stage 2: Build additional polygons from (leftover lines + existing polygon boundaries),
    so lines that close rings with polygon edges can form larger polygons.
    """
    if gdf_input.crs is None:
        raise ValueError("Input GeoDataFrame must have CRS.")

    projected_crs = gdf_input.estimate_utm_crs()
    work = gdf_input.to_crs(projected_crs).copy()

    line_mask = work.geometry.geom_type.isin(["LineString", "MultiLineString"])
    poly_mask = work.geometry.geom_type.isin(["Polygon", "MultiPolygon"])

    gdf_lines = work[line_mask].copy()
    gdf_polys_existing = work[poly_mask].copy()
    gdf_non_line_non_poly = work[~line_mask & ~poly_mask].copy()

    input_lines = []
    for geom in gdf_lines.geometry:
        input_lines.extend(_flatten_lines(geom))

    # Stage 1: line-only polygonization
    stage1_polys, stage1_leftovers, _ = _build_full_polygons(
        lines=input_lines,
        snap_tolerance_m=snap_tolerance_m,
        min_area_m2=min_area_m2,
    )

    # Stage 2: fuse leftover lines with polygon boundaries
    poly_boundaries = []
    for geom in gdf_polys_existing.geometry:
        valid_geom = _safe_make_valid(geom)
        for p in _flatten_polygons(valid_geom):
            poly_boundaries.append(p.boundary)

    stage2_lines = stage1_leftovers + poly_boundaries
    stage2_polys, stage2_leftovers, _ = _build_full_polygons(
        lines=stage2_lines,
        snap_tolerance_m=snap_tolerance_m,
        min_area_m2=min_area_m2,
    )

    existing_polys = []
    for geom in gdf_polys_existing.geometry:
        valid_geom = _safe_make_valid(geom)
        existing_polys.extend(_flatten_polygons(valid_geom))

    accepted_stage2 = []
    for p in stage2_polys:
        p = _safe_make_valid(p)
        if p is None or p.is_empty:
            continue

        is_duplicate = False
        for e in existing_polys:
            inter = _safe_intersection_area(p, e)
            if inter / max(p.area, 1e-9) >= dedupe_overlap_ratio:
                is_duplicate = True
                break
        if is_duplicate:
            continue

        overlaps_existing = [e for e in existing_polys if _safe_intersects(p, e)]
        if overlaps_existing:
            union_with_overlap = unary_union(overlaps_existing)
            if p.area >= union_with_overlap.area * area_growth_ratio:
                accepted_stage2.append(p)
        else:
            accepted_stage2.append(p)

    # Build output frames
    cols = list(gdf_input.columns)

    def _make_frame(geoms, source_name):
        if not geoms:
            return gpd.GeoDataFrame(columns=cols, crs=projected_crs, geometry="geometry")
        frame = gpd.GeoDataFrame({"geometry": geoms}, crs=projected_crs)
        if "Name" in cols:
            frame["Name"] = source_name
        if "Description" in cols:
            frame["Description"] = source_name
        if "layer" in cols:
            frame["layer"] = source_name
        for c in cols:
            if c not in frame.columns:
                frame[c] = None
        return frame[cols]

    frame_stage1_polys = _make_frame(stage1_polys, "Derived from linework")
    frame_stage2_polys = _make_frame(accepted_stage2, "Derived from polygon+line fusion")
    frame_leftovers = _make_frame(stage2_leftovers, "Leftover linework")

    gdf_final_proj = gpd.GeoDataFrame(
        pd.concat(
            [
                gdf_non_line_non_poly[cols],
                gdf_polys_existing[cols],
                frame_stage1_polys,
                frame_stage2_polys,
                frame_leftovers,
            ],
            ignore_index=True,
        ),
        crs=projected_crs,
        geometry="geometry",
    )

    stats = {
        "input_line_segments": len(input_lines),
        "stage1_polygons_from_lines": len(stage1_polys),
        "stage1_leftover_segments": len(stage1_leftovers),
        "stage2_polygons_from_polygon_plus_line": len(accepted_stage2),
        "stage2_leftover_segments": len(stage2_leftovers),
    }

    return gdf_final_proj.to_crs(gdf_input.crs), stats

gdf_final, conversion_stats = convert_lines_and_fuse_with_polygons(
    gdf_input=gdf_all,
    snap_tolerance_m=15,   # strict; try 10-15 if small gaps are still missed
    min_area_m2=25.0,
    dedupe_overlap_ratio=0.98,
    area_growth_ratio=1.05,
 )

print("Conversion stats:", conversion_stats)
print("Geometry mix after final fusion:")
print(gdf_final.geom_type.value_counts())

Conversion stats: {'input_line_segments': 61, 'stage1_polygons_from_lines': 40, 'stage1_leftover_segments': 147, 'stage2_polygons_from_polygon_plus_line': 3, 'stage2_leftover_segments': 159}
Geometry mix after final fusion:
LineString    159
Polygon        81
Point          34
Name: count, dtype: int64


In [56]:
import folium

In [ ]:
# Create folium map centered on India
india_center = [20.5937, 78.9629]
m = folium.Map(location=india_center, zoom_start=5, tiles="CartoDB positron")

gdf_map = gdf_all
    
gdf_map = gdf_map.to_crs(epsg=4326)  # Ensure it's in lat/lon for folium

# Define styles for different geometry types
styles = {
    'Polygon': {'fillColor': '#2d8659', 'color': '#000000', 'weight': 1, 'fillOpacity': 0.6},
    'MultiPolygon': {'fillColor': '#2d8659', 'color': '#000000', 'weight': 1, 'fillOpacity': 0.6},
    'LineString': {'color': '#ff6b6b', 'weight': 2, 'opacity': 0.8},
    'MultiLineString': {'color': '#ff6b6b', 'weight': 2, 'opacity': 0.8},
    'Point': {'color': '#4ecdc4', 'radius': 5, 'fillOpacity': 0.7}
}

# Add each geometry type as a separate feature group
if len(gdf_map) > 0:
    for geom_type in gdf_map.geometry.geom_type.unique():
        subset = gdf_map[gdf_map.geometry.geom_type == geom_type]
        
        fg = folium.FeatureGroup(name=f"{geom_type} ({len(subset)})", show=True)
        
        def style_fn(feature, gt=geom_type):
            return styles.get(gt, {'color': '#cccccc'})
        
        folium.GeoJson(
            subset.to_json(),
            style_function=style_fn,
            tooltip=folium.GeoJsonTooltip(
                fields=["Name", "Description", "layer"],
                aliases=["Name", "Description", "Layer"]
            ),
        ).add_to(fg)
        
        fg.add_to(m)
    
    print(f"\n✓ Added {len(gdf_map):,} reserve forest features to map")
else:
    print("\n⚠ No Reserve Forest features found")

# Add layer control
folium.LayerControl(collapsed=False).add_to(m)

# Save and display
output_file = "TN_OG_reserve_forests_map.html"
m.save(str(output_file))
print(f"\n✓ Map saved to: {output_file}")
print("Open the HTML file in your browser to view the interactive map")


✓ Added 274 reserve forest features to map

✓ Map saved to: TN_reserve_forests_map.html
Open the HTML file in your browser to view the interactive map
